#   **LangChain Custom Tool** (Part2)

- LangChain과 외부 API 통합

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

---

##  **사용자 정의 도구 (Custom Tool)**


- **사용자 정의 도구**는 개발자가 직접 설계하고 구현하는 **맞춤형 함수나 도구**를 의미

- LLM이 호출할 수 있는 **고유한 기능**을 정의하여 특정 작업에 최적화된 도구 생성 가능

- 개발자는 도구의 **입력값, 출력값, 기능**을 자유롭게 정의하여 유연한 확장성 확보

---

### 1. **외부 API 연동** 

- LangChain에서는 **사용자 정의 도구**를 통해 외부 API와의 연동이 가능

- **Tool** 클래스를 상속받아 필요한 기능을 구현하며, `name`과 `description`을 필수로 정의

- 도구는 단일 기능을 수행하는 **함수 형태**로 구현되며, 입력과 출력이 명확하게 설정함

`(1) Yahoo Finance API` 

In [ ]:
#  yfinance 설치 : pip install yfinance 또는 poetry add yfinance
import yfinance as yf

dat = yf.Ticker("MSFT")

In [ ]:
# 기업정보
dat.info

In [ ]:
# 주요 일정
dat.calendar

In [ ]:
# 2022년 1월 3일 ~ 4일의 데이터를 판다스 데이터프레임으로 출력 
result = dat.history(start="2022-01-03", end="2022-01-05") 
result

In [ ]:
# 인덱스 초기화 
result = result.reset_index()
result

In [ ]:
# 날짜 부분만 추출
result['Date'] = result['Date'].dt.strftime('%Y-%m-%d')

result

In [ ]:
# 데이터프레임을 딕셔너리로 변환
result_dict = result.to_dict(orient='records') 
result_dict

`(2) 데이터 연동 및 출력 포맷` 

In [ ]:
# 외부 API 연동하는 함수 (yfinance 사용)

from langchain_core.tools import ToolException
from typing import Dict, Optional
from datetime import datetime, timedelta
import yfinance as yf

def get_stock_price(symbol: str, date: Optional[str] = None) -> Dict:
    """yfiance 사용하여 특정 날짜의 주식의 가격 정보를 조회합니다."""

    if date and not is_valid_date(date):
        raise ToolException(f"잘못된 날짜 형식입니다: {date}")
    
    try:
        stock = yf.Ticker(symbol)
        # 특정 날짜의 주식 가격 정보 조회 
        if date:
            start = datetime.strptime(date, "%Y-%m-%d")
            end = start + timedelta(days=1)
            price = stock.history(start=start, end=end)

            # 가격 정보가 없으면 해날 날짜로부터 과거 5일간의 주식 가격 정보 조회
            if price.empty:
                end = start - timedelta(days=5)
                price = stock.history(start=end, end=start)

        # 특정 날짜가 없으면 최근 5일간의 주식 가격 정보 조회
        else:
            price = stock.history(period="5d")
            
        # 데이터프레임을 딕셔너리로 변환하여 반환 (가장 최근 날짜 데이터만 반환)
        df = price.reset_index()
        df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
        return df.to_dict(orient='records')[-1]
    
    except Exception as e:
        raise ToolException(str(e))
    

def is_valid_date(date_str: str) -> bool:
    try:
        datetime.strptime(date_str, '%Y-%m-%d')
        return True
    except ValueError:
        return False
    

# 함수 실행
result = get_stock_price("AAPL")

# 결과 출력
print(result)

In [ ]:
# 함수 실행 (날짜 지정)
result = get_stock_price("AAPL", "2025-01-03")
print(result)

`(3) StructuredTool 도구 변환` 

In [ ]:
from langchain_core.tools import StructuredTool

# StructuredTool로 도구 생성
stock_tool_basic = StructuredTool.from_function(
    func=get_stock_price,
    name="stock_price_basic",
    description="yfinance를 사용하여 주식 가격 정보를 조회하는 도구입니다.",
)

# 도구 실행 (정상)
result = stock_tool_basic.invoke({"symbol": "AAPL"})
print(result)

`(4) LLM 사용하여 도구 사용` 

In [ ]:
from langchain_openai import ChatOpenAI

# OpenAI GPT-4.1-mini 모델 사용
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 도구 바인딩
llm_with_tool = llm.bind_tools([stock_tool_basic])

# 도구 호출
result = llm_with_tool.invoke("애플의 주식 가격을 알려줘")
pprint(result.tool_calls)
print("-"*100)

# ToolCall을 도구에 전달하여 결과 확인
tool_call = result.tool_calls[0]
result = stock_tool_basic.invoke(tool_call)

print(result)

In [ ]:
# 도구 호출 (날짜 지정)
result = llm_with_tool.invoke("애플의 2025년 1월 3일 주식 가격을 알려줘")
pprint(result.tool_calls)

# ToolCall을 도구에 전달하여 결과 확인
for tool_call in result.tool_calls:
    result = stock_tool_basic.invoke(tool_call)
    print(result)

In [ ]:
from langchain.agents import create_agent

# 도구 실행 에이전트 생성 
stock_agent = create_agent(
    model=llm,
    tools=[stock_tool_basic],
    system_prompt="당신은 주식 정보를 제공하는 AI 어시스턴트입니다."
)

# 도구 실행 에이전트 사용
result = stock_agent.invoke(
    {"messages": [{"role": "user", "content": "애플의 2025년 1월 3일 주식 가격을 알려줘"}]}
)

pprint(result['messages'])


In [ ]:
# 도구 실행 에이전트 생성 (return_direct=True)
stock_tool_basic.return_direct = True
stock_agent = create_agent(
    model=llm,
    tools=[stock_tool_basic],
    system_prompt="당신은 주식 정보를 제공하는 AI 어시스턴트입니다."
)

# 도구 실행 에이전트 사용
result = stock_agent.invoke(
    {"messages": [{"role": "user", "content": "애플의 2025년 1월 3일 주식 가격을 알려줘"}]}
)

pprint(result['messages'])

---

### [실습]

- yfinance 사용하여 특정 기간(시작일 ~ 종료일) 동안의 거래 데이터를 가져오는 도구를 정의 

- **Hint**: history 메소드의 start, end 속성에 날짜 지정 

In [ ]:
# 여기에 코드를 작성하세요.

---

### 2. **도구 에러 처리** 

> **⚠️ LangChain v1.0 업데이트**  
> - 도구 에러 처리는 `handle_tool_error` 파라미터로 계속 사용 가능
> - 추가로 **미들웨어(Middleware)** 를 통한 고급 에러 처리도 지원
> - 미들웨어는 더 유연한 에러 처리와 재시도 로직 구현 가능

- **handle_tool_error** 매개변수를 통해 도구의 오류를 체계적으로 관리

- 에이전트와 도구 사이의 **에러 처리 로직**을 명확하게 정의할 수 있음

- LangChain이 제공하는 **기본 에러 처리 방식**으로 안정적인 실행 보장

- 도구 실행 중 발생하는 **예외 상황**을 효과적으로 제어 가능


In [ ]:
# 함수 실행 (날짜 오류)
result = get_stock_price("AAPL", "22-01-01")
print(result)

In [ ]:
# 도구 실행 (날짜 오류)
result = stock_tool_basic.invoke({"symbol": "AAPL", "date": "22-01-01"})
print(result)

`(1) 기본 에러 처리`
   - *handle_tool_error=True*를 사용
   - ToolException의 메시지를 그대로 반환

In [ ]:
from langchain_core.tools import StructuredTool

# StructuredTool로 도구 생성
stock_tool_basic = StructuredTool.from_function(
    func=get_stock_price,
    name="stock_price_basic",
    handle_tool_error=True  # 기본 에러 메시지 반환
)

# 도구 실행 (정상)
result = stock_tool_basic.invoke({"symbol": "AAPL"})
print(result)

In [ ]:
# 도구 실행 (날짜 오류)
result = stock_tool_basic.invoke({"symbol": "AAPL", "date": "22-01-01"})
print(result)

`(2) 커스텀 에러 메시지`
   - *handle_tool_error*="에러 메시지"를 사용
   - 모든 에러에 대해 동일한 메시지 반환

In [ ]:
# StructuredTool로 도구 생성
stock_tool_custom = StructuredTool.from_function(
    func=get_stock_price,
    name="stock_price_custom",
    handle_tool_error="유효한 날짜 형식(YYYY-MM-DD)이 아닙니다. 다시 입력해주세요."
)

# 도구 실행 (정상)
result = stock_tool_custom.invoke({"symbol": "AAPL"})
print(result)

In [ ]:
# 도구 실행 (날짜 오류)
result = stock_tool_custom.invoke({"symbol": "AAPL", "date": "22-02-02"})
print(result)

`(3) 커스텀 에러 처리 함수`
   - handle_tool_error에 함수 전달
   - 에러 타입에 따라 다른 처리 가능

In [ ]:
# 커스텀 에러 처리 함수
def custom_error_handler(error: Exception) -> str:
    if "잘못된 날짜" in str(error):
        return f"날짜 형식 오류: {str(error)}. YYYY-MM-DD 형식으로 입력해주세요."
    else:
        return f"도구 실행 중 오류가 발생했습니다: {str(error)}"

# StructuredTool로 도구 생성 (커스텀 에러 처리 함수)
stock_tool_custom_fn = StructuredTool.from_function(
    func=get_stock_price,
    name="stock_price_custom_fn",
    handle_tool_error=custom_error_handler
)

In [ ]:
# 도구 실행 (정상)
result = stock_tool_custom_fn.invoke({"symbol": "AAPL"})
print(result)

In [ ]:
# 도구 실행 (날짜 오류)
result = stock_tool_custom_fn.invoke({"symbol": "AAPL", "date": "22-01-01"})
print(result)

`(4) 미들웨어를 통한 고급 에러 처리` 

- LangChain v1.0에서 추가된 기능

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage

@wrap_tool_call
def handle_tool_errors(request, handler):
    """도구 실행 에러를 처리하는 미들웨어"""
    try:
        return handler(request)
    except Exception as e:
        # 커스텀 에러 메시지 반환
        return ToolMessage(
            content=f"도구 실행 중 오류 발생: {str(e)}. 입력값을 확인해주세요.",
            tool_call_id=request.tool_call["id"]
        )

# 미들웨어를 적용한 에이전트 생성
agent = create_agent(
    model=llm,
    tools=[stock_tool_basic],
    middleware=[handle_tool_errors],
    system_prompt="당신은 주식 정보를 제공하는 AI 어시스턴트입니다."
)

# 에이전트 실행 (오류 발생 시에도 안전하게 처리)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "애플의 202-01-01 주식 가격을 알려줘"}]}
)

pprint(result['messages'])

In [ ]:
for msg in result['messages']:
    msg.pretty_print()

---

### [실습]

- 도구 에러 처리 유형을 살펴 보고, 직접 수정하여 적용해봅니다. 

In [ ]:
# 여기에 코드를 작성하세요. 

---

### 3. **BaseTool 상속** 

- **BaseTool**을 상속하여 더 복잡한 도구 구현 가능

- `_run` 메소드에서 **동기 실행 로직** 구현

- `_arun` 메소드에서 **비동기 실행 로직** 구현 (선택사항)

- Pydantic 모델로 **입력 스키마**를 명확하게 정의

`(1) 입력 스키마 정의`

In [ ]:
from pydantic import BaseModel, Field

class StockPriceInput(BaseModel):
    """주식 가격 조회 도구의 입력 스키마"""
    symbol: str = Field(description="주식 심볼 (예: AAPL, TSLA)")
    date: str = Field(description="조회할 날짜 (YYYY-MM-DD 형식)")

`(2) BaseTool 상속하여 도구 구현`

In [ ]:
from langchain_core.tools import BaseTool
from langchain_core.callbacks import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)

class StockPriceTool(BaseTool):
    name: str = "StockPrice"
    description: str = "yfinance를 사용하여 특정 날짜의 주식 가격 정보를 조회합니다."
    args_schema: type[BaseModel] = StockPriceInput
    return_direct: bool = False

    def _run(
        self,
        symbol: str,
        date: str,
        run_manager: Optional[CallbackManagerForToolRun] = None
    ) -> dict:
        """도구를 동기적으로 실행합니다."""
        
        # 날짜 유효성 검사
        if not is_valid_date(date):
            raise ToolException(f"잘못된 날짜 형식입니다: {date}")

        try:
            stock = yf.Ticker(symbol)
            start = datetime.strptime(date, "%Y-%m-%d")
            end = start + timedelta(days=1)
            
            price = stock.history(start=start, end=end)

            # 가격 정보가 없으면 최근 거래일 데이터 조회
            if price.empty:
                price = stock.history(period="5d")
                if price.empty:
                    raise ToolException(f"{symbol}에 대한 주식 데이터를 찾을 수 없습니다.")
                
                price = price.reset_index()
                price['Date'] = price['Date'].dt.strftime('%Y-%m-%d')
                result = price.to_dict(orient="records")[-1]
                
                return {
                    "symbol": symbol,
                    "date": result.get("Date"),
                    "open": result.get("Open"),
                    "high": result.get("High"),
                    "low": result.get("Low"),
                    "close": result.get("Close"),
                    "volume": result.get("Volume"),
                    "comment": f"{date} 날짜의 데이터가 없어 최근 거래일인 {result.get('Date')} 날짜의 주식 가격 정보 조회"
                }

            # 데이터프레임을 딕셔너리로 변환하여 반환 (가장 최근 날짜 데이터만 반환)
            price = price.reset_index()
            price['Date'] = price['Date'].dt.strftime('%Y-%m-%d')
            result = price.to_dict(orient="records")[-1]
            return {
                "symbol": symbol,
                "date": result.get("Date"),
                "open": result.get("Open"),
                "high": result.get("High"),
                "low": result.get("Low"),
                "close": result.get("Close"),
                "volume": result.get("Volume"),
                "comment": f"{result.get('Date')} 날짜의 주식 가격 정보 조회"
            }
                
        except Exception as e:
            raise ToolException(str(e))

    
    async def _arun(
        self,
        symbol: str,
        date: str,
        run_manager: Optional[AsyncCallbackManagerForToolRun] = None
    ) -> dict:
        """도구를 비동기적으로 실행합니다."""
        return self._run(symbol, date, run_manager.get_sync() if run_manager else None)

`(3) 도구 실행`

In [ ]:
# 도구 생성
stock_tool = StockPriceTool()

# 도구 속성
print(stock_tool.name)
print(stock_tool.description)
print(stock_tool.args_schema)
print(stock_tool.return_direct)

In [ ]:
# 도구 실행 (거래일이 없는 날짜)
result = stock_tool.invoke({"symbol": "TSLA", "date": "2023-12-25"})
pprint(result)


In [ ]:
# 도구 실행 (거래일이 있는 날짜)
result = stock_tool.invoke({"symbol": "AAPL", "date": "2022-01-03"})
pprint(result)


In [ ]:
# LLM에 도구 바인딩하여 사용
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tool = llm.bind_tools([stock_tool])

# 도구 호출
result = llm_with_tool.invoke("애플의 2022년 1월 3일 주식 가격을 알려줘")
pprint(result.tool_calls)


In [ ]:
# ToolCall을 도구에 전달하여 결과 확인
for tool_call in result.tool_calls:
    msg = stock_tool.invoke(tool_call)
    pprint(msg.content)


In [ ]:
# 도구 실행 (거래일이 없는 날짜)
result = llm_with_tool.invoke("테슬라의 2023년 12월 25일 주식 가격을 알려줘")
pprint(result.tool_calls)


In [ ]:
# ToolCall을 도구에 전달하여 결과 확인
for tool_call in result.tool_calls:
    msg = stock_tool.invoke(tool_call)
    pprint(msg.content)

---

### [실습]

- BaseTool 상속받아서 도구 정의하는 과정을 살펴 보고, 직접 수정하여 적용해봅니다. 

In [ ]:
# 여기에 코드를 작성하세요. 

---

### 4. **QA 체인 구성** 

- **QA 체인**을 구성하여 질의응답 시스템을 체계화할 수 있음

- **도구 사용**을 통해 사용자가 원하는 정보를 컨텍스트로 활용 가능 

In [ ]:
# 도구의 이름 속성 확인 
stock_tool.name

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain
from langchain_openai import ChatOpenAI
from datetime import datetime

# LLM 모델 인스턴스를 생성
llm = ChatOpenAI(model="gpt-4o-mini")

# 두 도구를 LLM에 바인딩
llm_with_tools = llm.bind_tools(tools=[stock_tool])

# 오늘 날짜 설정
today = datetime.today().strftime("%Y-%m-%d")

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate([
    ("system", f"당신은 도움이 되는 AI 어시스턴트입니다. 오늘 날짜는 {today}입니다."),
    ("human", "{user_input}"),
    ("placeholder", "{messages}"),
])

# LLM 체인 생성
llm_chain = prompt | llm_with_tools

@chain
def stock_analysis_chain(user_input: str, config: RunnableConfig):
    input_ = {"user_input": user_input}
    ai_msg = llm_chain.invoke(input_, config=config)

    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        try:
            if tool_call["name"] == "StockPrice":
                tool_result = stock_tool.invoke(tool_call)
            else:
                print(f"알 수 없는 도구 호출: {tool_call['name']}")
                continue

            # tool message 출력
            print("Tool Name: ", tool_call["name"])
            print(tool_result)
            print("-"*200)

            tool_msgs.append(tool_result)
        except Exception as e:
            print(f"{tool_call['name']} 도구 호출 중 에러 발생: {e}")
    
    return llm_chain.invoke({**input_, "messages": [ai_msg, *tool_msgs]}, config=config)

# 체인 실행
query = "애플의 1월말 주가는 얼마인가요?"
response = stock_analysis_chain.invoke(query)

print(response)



In [ ]:
print(response.content)

---

### [심화] **Naver 개발자 API** 를 도구로 사용

#### **환경 설정**

- 네이버 개발자 API(https://developers.naver.com/)에서 인증 권한 취득 (회원 가입 및 애플리케이션 등록 필요)
- 환경변수(.env)를 등록합니다. (**NAVER_CLIENT_ID**, **NAVER_CLIENT_SECRET**)
- 아래 정의된 네이버 뉴스 검색 도구(naver_news_search)를 사용하여 다음 과정을 수행합니다. 

In [ ]:
# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import requests, os
from langchain_core.tools import tool
from typing import Dict

@tool
def naver_news_search(
    query: str,
    ) -> Dict[Dict, int]:
    """네이버 검색 API를 사용하여 뉴스 검색 결과를 조회합니다.

    Args:
        query (str): 검색어

    Returns:
        Dict[Dict, int]: 검색 결과와 상태 코드  
    """


    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": os.getenv("NAVER_CLIENT_ID"),
        "X-Naver-Client-Secret": os.getenv("NAVER_CLIENT_SECRET")
    }
    params = {"query": query}

    response = requests.get(url, headers=headers, params=params)

    return {
        "data": response.json(),
        "status_code": int(response.status_code)
    }  #type: ignore

---

#### **문제 1**: 도구 실행 체인 구성

1. 도구 설정
    - 도구들(naver_news_search, stock_tool)을 리스트로 구성
    - 도구 이름을 키로 하는 맵(tool_map) 생성

2. 데코레이터 활용
    - @chain 데코레이터로 일반 함수를 체인으로 변환
    - tool_router: 도구 이름에 따라 적절한 도구 선택

3. 체인 구성
    ```python
    tool_chain = (
        llm_with_tools    # 어떤 도구를 사용할지 결정 (LLM 모델이 도구 호출을 처리)
        | RunnableLambda(lambda x: x.tool_calls)  # 도구 호출을 추출
        | tool_router.map()   # 도구 호출 라우팅
    )
    ```

4. 동작 순서
    1. LLM이 상황에 맞는 도구 선택
    2. 해당 도구 실행

In [ ]:
# 여기에 코드를 작성하세요.

---

#### **문제 2**: 주식 분석 체인 구성

1. 기본 설정
    ```python
    llm = ChatOpenAI(model="gpt-4.1-mini")
    llm_with_tools = llm.bind_tools([stock_tool, naver_news_search])
    ```

2. 프롬프트 구성
    - 시스템/사용자 메시지 포함
    - 도구 실행 결과를 위한 placeholder 설정

3. 분석 체인 동작
    - 도구 체인으로 주가/뉴스 정보 수집
    - 도구 응답을 메시지 형식으로 변환 
    - LLM으로 최종 분석 응답 생성

4. 데이터 흐름
     - 사용자 입력 -> 도구 실행 -> 결과 포맷팅 -> LLM 분석 -> 최종 응답


In [ ]:
# 여기에 코드를 작성하세요.